# Pancake Stayman

**Research question:** When both opener and responder hold flat hands, does using Stayman gain or lose compared to simply raising to 3NT?

"Pancake" hands are the flattest possible — 4333 distributions. These are the hands where Stayman is *least* expected to gain, because:
- The 4–4 major suit fit offers little ruffing value when both hands are completely flat.
- Declarer in a major suit contract must lose a trump trick on any 4–2 break, whereas 3NT has no such vulnerability.

We quantify the effect using a double-dummy simulation.

## Setup

- **South** is a 1NT opener: balanced 15–17 HCP (`any 4333 + any 4432 + any 5332`).
- **North** is a game-forcing responder with no slam interest: 4333 or 3433 shape, 9–14 HCP.
  - `4333`: North has exactly 4 spades (will bid Stayman looking for a spade fit).
  - `3433`: North has exactly 4 hearts (will bid Stayman looking for a heart fit).

### Auction outcomes

| Strategy | Contract |
|---|---|
| **Raise** | 3NT–S always |
| **Stayman** | 4♥–S if both hold 4+ hearts; 4♠–S if both hold 4+ spades; else 3NT–S |

In [ ]:
import bridgepandas as bp
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

## 1. Generate deals

We use accept/reject sampling with scalar `Hand` properties for speed.
Generation takes roughly 30 seconds for 1000 deals.

In [ ]:
_SOUTH_SHAPES   = {(5, 3, 3, 2), (4, 4, 3, 2), (4, 3, 3, 3)}  # any 5332 / 4432 / 4333
_NORTH_PATTERNS = {(4, 3, 3, 3), (3, 4, 3, 3)}                 # 4 spades or 4 hearts

def accept(deal):
    s, n = deal.south, deal.north
    return (
        s.shape in _SOUTH_SHAPES and 15 <= s.hcp <= 17
        and n.pattern in _NORTH_PATTERNS and 9 <= n.hcp <= 14
    )

N = 1000
df = bp.random_deals(N, accept=accept, seed=42)
print(f"{len(df)} deals generated")

## 2. Verify constraints

In [ ]:
pd.DataFrame({
    'south_hcp':    df['south'].hcp,
    'south_spades': df['south'].spades,
    'south_hearts': df['south'].hearts,
    'north_hcp':    df['north'].hcp,
    'north_spades': df['north'].spades,
    'north_hearts': df['north'].hearts,
}).describe().round(2)

In [ ]:
south_shapes = pd.Series([df.iloc[i].south.shape for i in range(N)])
print("South shapes:")
print(south_shapes.value_counts().to_string())
print()
north_has_4s = df['north'].spades == 4
print(f"North 4333 (4 spades): {north_has_4s.sum()}  ({north_has_4s.mean():.1%})")
print(f"North 3433 (4 hearts): {(~north_has_4s).sum()}  ({(~north_has_4s).mean():.1%})")

## 3. Determine Stayman contracts

North always has exactly one 4-card major. Stayman finds a fit if and only if South also holds 4+ cards in that suit.

In [ ]:
heart_fit = (df['north'].hearts >= 4) & (df['south'].hearts >= 4)
spade_fit = (df['north'].spades >= 4) & (df['south'].spades >= 4)

stayman_contracts = pd.Series(
    np.where(heart_fit, '4H-S', np.where(spade_fit, '4S-S', '3N-S'))
)

no_fit = ~heart_fit & ~spade_fit
print("Stayman auction outcomes:")
print(f"  4♥–S (heart fit): {heart_fit.sum():4d}  ({heart_fit.mean():.1%})")
print(f"  4♠–S (spade fit): {spade_fit.sum():4d}  ({spade_fit.mean():.1%})")
print(f"  3NT–S (no fit):   {no_fit.sum():4d}  ({no_fit.mean():.1%})")

## 4. Double-dummy scoring

We use neither-vulnerable throughout. The `_dds` cache means the 3NT trick
count is computed once and reused when Stayman also lands in 3NT.

In [ ]:
VULN = '-'  # neither vulnerable

bp.add_dds(df, '3N-S',           'score_3nt',     VULN)
bp.add_dds(df, stayman_contracts, 'score_stayman', VULN)

df[['score_3nt', 'score_stayman']].head(10)

## 5. IMP differential

`imp_diff > 0` means 3NT scored better on that board; `imp_diff < 0` means Stayman scored better.

In [ ]:
df['imp_diff'] = (df['score_3nt'] - df['score_stayman']).map(bp.scorediff_imps)
df['outcome']  = np.where(heart_fit, '4H fit', np.where(spade_fit, '4S fit', 'No fit'))

df[['score_3nt', 'score_stayman', 'imp_diff', 'outcome']].head(10)

## 6. Statistical analysis

In [ ]:
mean_diff       = df['imp_diff'].mean()
std_diff        = df['imp_diff'].std()
se              = std_diff / np.sqrt(N)
t_stat, p_value = stats.ttest_1samp(df['imp_diff'], 0)
ci_lo, ci_hi    = mean_diff - 1.96 * se, mean_diff + 1.96 * se

print("IMP differential  (positive = 3NT better, negative = Stayman better)")
print(f"  n         = {N}")
print(f"  mean      = {mean_diff:+.3f} IMPs")
print(f"  std dev   = {std_diff:.3f}")
print(f"  std error = {se:.3f}")
print(f"  95% CI    = ({ci_lo:+.3f}, {ci_hi:+.3f})")
print(f"  t         = {t_stat:.3f}")
print(f"  p-value   = {p_value:.4f}  (two-sided, H\u2080: mean = 0)")
print()
if p_value < 0.05:
    winner = '3NT' if mean_diff > 0 else 'Stayman'
    print(f"\u2192 {winner} is significantly better (p < 0.05).")
else:
    print(f"\u2192 No significant difference detected (p = {p_value:.3f}).")

## 7. Breakdown by auction outcome

In [ ]:
breakdown = (
    df.groupby('outcome')['imp_diff']
    .agg(count='count', mean='mean', std='std')
    .assign(se=lambda d: d['std'] / np.sqrt(d['count']))
)
breakdown[['count', 'mean', 'se']].round(3)

## 8. Distribution of IMP differentials

In [ ]:
colors = {'4H fit': 'tomato', '4S fit': 'steelblue', 'No fit': 'silver'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall distribution
ax = axes[0]
ax.hist(df['imp_diff'], bins=range(-15, 17), edgecolor='white', color='steelblue', alpha=0.8)
ax.axvline(0,         color='black', linewidth=0.8, linestyle='--')
ax.axvline(mean_diff, color='red',   linewidth=1.5, label=f'mean = {mean_diff:+.2f} IMPs')
ax.set_xlabel('IMP differential  (3NT \u2212 Stayman)')
ax.set_ylabel('Frequency')
ax.set_title('Overall')
ax.legend()

# Mean IMP diff by outcome with 95% CI error bars
ax = axes[1]
outcomes    = breakdown.index.tolist()
means       = breakdown['mean'].values
errs        = 1.96 * breakdown['se'].values
bar_colors  = [colors[o] for o in outcomes]
bars = ax.bar(outcomes, means, yerr=errs, capsize=5,
              color=bar_colors, edgecolor='white', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_ylabel('Mean IMP differential  (3NT \u2212 Stayman)')
ax.set_title('By auction outcome  (95% CI)')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2,
            m + np.sign(m) * 0.15,
            f'{m:+.2f}', ha='center',
            va='bottom' if m >= 0 else 'top', fontsize=9)

fig.suptitle('Pancake Stayman: does Stayman gain on flat hands?', fontsize=13)
plt.tight_layout()
plt.show()